### **Configuração Pasta Raiz**

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

# Adiciona a raiz do projeto ao path do Python para que as importações funcionem
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# print(f"Raiz do projeto definida como: {PROJECT_ROOT}")

Raiz do projeto definida como: c:\Users\caio\OneDrive\Documents\GitHub\nba-draft-prediction


### **Bibliotecas**

In [2]:
import pandas as pd
import src.utils.converters as convert
import src.utils.features as features
from src.config import FILES

### **Leitura da base de dados**

In [3]:
dfs = {}

for key, filename in FILES.items():
    file_path = f'../{filename}'
    
    try:
        dfs[key] = pd.read_csv(file_path)
        
        print(f"✅ {key.ljust(15)}: Carregado ({len(dfs[key])} linhas)")
        
    except FileNotFoundError:
        print(f"❌ {key.ljust(15)}: Arquivo não encontrado em '{file_path}'")
    except Exception as e:
        print(f"❌ {key.ljust(15)}: Erro - {e}")

✅ draft          : Carregado (1542 linhas)
✅ nba            : Carregado (7009 linhas)
✅ college_adv    : Carregado (3260 linhas)
✅ college_per_min: Carregado (3260 linhas)
✅ college_player_metadata: Carregado (1183 linhas)
✅ mock           : Carregado (1548 linhas)
✅ combine_anthro : Carregado (1795 linhas)
❌ combine_strength_agility: Arquivo não encontrado em '../data/nba_combine_strength_agility.csv'


### **Executando cortes nas bases**

#### **Draft**

In [4]:
dfs['draft'].head()

,Rk,Pk,Tm,Player,College,Yrs,G,MP,PTS,TRB,...,RPG,APG,WS,WS_per_48,BPM,VORP,Year,Player_Link,Player_ID,College_Link
0,1,1,NJN,Kenyon Martin,Cincinnati,15.0,757.0,23134.0,9325.0,5159.0,...,6.8,1.9,48.0,0.100,0.1,12.1,2000,https://www.basketball-reference.com/players/m...,martike01,https://www.sports-reference.com/cbb/players/k...
1,2,2,VAN,Stromile Swift,LSU,9.0,547.0,10804.0,4582.0,2535.0,...,4.6,0.5,21.3,0.095,-1.6,1.1,2000,https://www.basketball-reference.com/players/s...,swiftst01,https://www.sports-reference.com/cbb/players/s...
2,3,3,LAC,Darius Miles,NaN,7.0,446.0,11730.0,4507.0,2190.0,...,4.9,1.9,9.5,0.039,-1.0,3.0,2000,https://www.basketball-reference.com/players/m...,milesda01,NaN
3,4,4,CHI,Marcus Fizer,Iowa State,6.0,289.0,6032.0,2782.0,1340.0,...,4.6,1.2,2.7,0.022,-3.7,-2.6,2000,https://www.basketball-reference.com/players/f...,fizerma01,https://www.sports-reference.com/cbb/players/m...
4,5,5,ORL,Mike Miller,Florida,17.0,1032.0,27812.0,10973.0,4376.0,...,4.2,2.6,60.7,0.105,0.8,19.8,2000,https://www.basketball-reference.com/players/m...,millemi01,https://www.sports-reference.com/cbb/players/m...


In [5]:
df_draft = dfs['draft'][['Player_ID','Player','Year','College','Pk']].copy()

In [6]:
df_draft.rename(columns={
    'Player': 'player_name', 
    'Pk': 'draft_pick',
    'Year': 'draft_year',
    'College': 'College_Name'
}, inplace=True)

df_draft.columns = df_draft.columns.str.lower()

In [7]:
df_draft.head()

,player_id,player_name,draft_year,college_name,draft_pick
0,martike01,Kenyon Martin,2000,Cincinnati,1
1,swiftst01,Stromile Swift,2000,LSU,2
2,milesda01,Darius Miles,2000,NaN,3
3,fizerma01,Marcus Fizer,2000,Iowa State,4
4,millemi01,Mike Miller,2000,Florida,5


#### **Estatísticas Avançadas da NBA**

In [8]:
dfs['nba'].head()

,Season,Age,Team,Lg,Pos,G,GS,MP,PER,TS_pct,...,OWS,DWS,WS,WS_per_48,OBPM,DBPM,BPM,VORP,Awards,Player_ID
0,2000-01,23,NJN,NBA,PF,68.0,68.0,2272.0,13.4,0.472,...,-0.4,2.5,2.1,0.044,-2.1,0.4,-1.7,0.2,ROY-2,martike01
1,2001-02,24,NJN,NBA,PF,73.0,73.0,2504.0,14.5,0.503,...,0.6,4.2,4.9,0.093,-1.0,1.4,0.4,1.5,NaN,martike01
2,2002-03,25,NJN,NBA,PF,77.0,77.0,2628.0,16.9,0.511,...,1.8,5.3,7.1,0.129,-0.2,1.2,1.0,2.0,DPOY-12,martike01
3,2003-04,26,NJN,NBA,PF,65.0,62.0,2252.0,18.7,0.528,...,1.5,5.1,6.7,0.142,0.1,1.8,1.9,2.2,"DPOY-9,AS",martike01
4,2004-05,27,DEN,NBA,PF,70.0,67.0,2272.0,17.5,0.521,...,1.7,3.6,5.3,0.111,0.1,1.0,1.1,1.8,NaN,martike01


In [9]:
df_nba_stats = dfs['nba'][['Player_ID','Age','VORP','Season']].copy()

df_nba_stats.columns = df_nba_stats.columns.str.lower()

In [10]:
df_nba_stats.head()

,player_id,age,vorp,season
0,martike01,23,0.2,2000-01
1,martike01,24,1.5,2001-02
2,martike01,25,2.0,2002-03
3,martike01,26,2.2,2003-04
4,martike01,27,1.8,2004-05


##### **Aplicação de limpeza e filtros**

In [11]:
# 1. Classificar e Filtrar: Queremos apenas as primeiras 5 temporadas de cada jogador.

df_nba_stats['season_int'] = df_nba_stats['season'].str.split('-').str[0].astype(int)

In [12]:
# Agrupa por jogador, ordena por ano, e pega apenas as 5 primeiras linhas
first_five_years = (
    df_nba_stats
    .sort_values(by=['player_id', 'season_int'])
    .groupby('player_id')
    .head(5)
)

In [13]:
# 2. Calcular o VORP Médio (Métrica Alvo)
# Calcular a soma total de VORP nos 5 primeiros anos
df_nba_stats = (
    first_five_years
    .groupby('player_id')
    .agg(
        vorp_5y_sum=('vorp', 'sum'),  # Soma o VORP dos 5 anos
        rookie_age=('age', 'min'),     # Pega a menor idade (Idade de estreia)
        seasons_played=('season', 'count') # Conta quantas temporadas ele jogou
    )
    .round({'vorp_5y_sum': 2})
    .reset_index() 
)

In [ ]:
seasons_played

In [14]:
df_nba_stats.sort_values(by='vorp_5y_sum', ascending=False)

,player_id,vorp_5y_sum,rookie_age,seasons_played
770,paulch01,32.0,20,5
272,duranke01,23.7,19,5
230,davisan02,21.4,19,5
997,wadedw01,21.1,22,5
974,townska01,21.1,20,5
...,...,...,...,...
664,mclembe01,-3.1,20,5
967,thornal01,-3.2,24,5
126,brookdi01,-3.6,22,5
675,mihmch01,-3.8,21,5


#### **Estatísticas Avançadas do College**

In [15]:
dfs['college_adv'].head()

,Season,Tm,Conf,Class,Pos,G,GS,MP,TS_pct,3PAr,...,STL_pct,BLK_pct,USG_pct,PProd,ORB_pct,PER,DRB_pct,OBPM,DBPM,BPM
0,1996-97,Cincinnati,CUSA,FR,C,22,3,233,0.562,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1997-98,Cincinnati,CUSA,SO,C,30,30,858,0.601,0.005,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1998-99,Cincinnati,CUSA,JR,C,33,33,900,0.575,0.004,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1999-00,Cincinnati,CUSA,SR,C,31,31,909,0.601,0.018,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1998-99,Louisiana State,SEC,FR,F,16,7,319,0.452,0.073,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
df_college_adv_init = dfs['college_adv'][['Player_ID','Season','TS_pct','3PAr','FTr','USG_pct','OWS','DWS','WS_per_40']].copy()

In [17]:
df_college_adv_init.rename(columns={
    'TS_pct': 'true_shooting_pct', 
    '3PAr': 'three_point_attempt_rate',
    'FTr': 'free_throw_attempt_rate',
    'USG_pct': 'usage_pct',
    'OWS': 'offensive_win_shares',
    'DWS': 'defensive_win_shares',
    'WS_per_40': 'win_shares_per_40min'
}, inplace=True)

df_college_adv_init.columns = df_college_adv_init.columns.str.lower()

In [18]:
df_college_adv_init.head()

,player_id,season,true_shooting_pct,three_point_attempt_rate,free_throw_attempt_rate,usage_pct,offensive_win_shares,defensive_win_shares,win_shares_per_40min
0,martike01,1996-97,0.562,0.0,0.8,NaN,0.3,1.5,0.316
1,martike01,1997-98,0.601,0.005,0.53,NaN,2.0,5.0,0.326
2,martike01,1998-99,0.575,0.004,0.359,NaN,2.0,4.8,0.303
3,martike01,1999-00,0.601,0.018,0.53,NaN,3.9,5.1,0.398
4,swiftst01,1998-99,0.452,0.073,0.455,NaN,-0.1,0.8,0.089


##### **Aplicação de limpeza e filtros**

In [19]:
# 1. Ordenar: Primeiro por Jogador, depois por Temporada (do mais antigo para o mais novo)
df_sorted = df_college_adv_init.sort_values(by=['player_id', 'season'], ascending=[True, True])

# 2. Filtrar: Remove duplicatas de ID, mantendo APENAS a última linha (a temporada mais recente)
df_college_adv = df_sorted.drop_duplicates(subset=['player_id'], keep='last').copy()

# 3. Drop Column: Remove coluna 'Season' que não é mais necessária
df_college_adv.drop('season', axis=1, inplace=True)

In [20]:
df_college_adv.head()

,player_id,true_shooting_pct,three_point_attempt_rate,free_throw_attempt_rate,usage_pct,offensive_win_shares,defensive_win_shares,win_shares_per_40min
2607,achiupr01,0.534,0.108,0.507,27.7,1.5,2.9,0.187
740,ackeral01,0.564,0.337,0.26,22.2,2.5,1.5,0.14
1677,acyqu01,0.638,0.018,0.635,19.0,3.5,2.0,0.198
872,adamsha01,0.507,0.133,0.194,28.2,1.2,2.6,0.15
1897,adamsjo01,0.603,0.34,0.441,26.5,4.2,2.2,0.237


#### **Estatísticas Game per Min do College**

In [21]:
dfs['college_per_min']

,Season,Tm,Conf,Class,Pos,G,GS,MP,FG,FGA,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,Player_ID
0,1996-97,Cincinnati,CUSA,FR,C,22,3,233,4.5,6.9,...,NaN,12.7,1.7,2.4,4.1,2.7,8.8,10.6,NaN,martike01
1,1997-98,Cincinnati,CUSA,SO,C,30,30,858,5.8,9.2,...,NaN,12.4,1.9,1.4,3.9,2.4,5.1,13.9,NaN,martike01
2,1998-99,Cincinnati,CUSA,JR,C,33,33,900,6.3,11.0,...,NaN,10.1,2.2,1.6,3.5,3.0,4.8,14.8,NaN,martike01
3,1999-00,Cincinnati,CUSA,SR,C,31,31,909,9.7,17.1,...,9.9,13.2,1.8,1.9,4.7,2.5,3.1,25.7,"AP POY-1,WA-1,AA-1,NA-1",martike01
4,1998-99,Louisiana State,SEC,FR,F,16,7,319,5.6,13.8,...,NaN,8.7,0.6,1.9,4.4,5.8,5.4,15.2,NaN,swiftst01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3255,2024-25,VCU,A-10,SR,G,35.0,35.0,1148.0,5.3,12.3,...,5.0,7.1,4.8,2.2,0.1,2.1,2.4,18.3,NaN,shulgma01
3256,2021-22,Tennessee,SEC,FR,G,27,0,123,2.3,5.5,...,2.9,6.5,2.6,3.6,0.7,2.9,6.8,6.2,NaN,mashaja01
3257,2022-23,Tennessee,SEC,SO,G,36,13,653,3.9,9.2,...,2.6,5.2,3.4,3.1,0.6,2.8,4.0,10.4,NaN,mashaja01
3258,2023-24,Tennessee,SEC,JR,G,36,8,644,3.5,7.8,...,3.7,6.9,2.9,1.9,0.7,1.4,4.8,10.1,NaN,mashaja01


In [22]:
df_college_per_min_init = dfs['college_per_min'][['Player_ID','Season', 'Conf', 'Class', 'G', 'GS', 'MP',  '3P', '3PA', 'eFG_pct', 'FTA', 'FT_pct', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'Awards', '3P_pct' ]]

In [23]:
df_college_per_min_init.head()

,Player_ID,Season,Conf,Class,G,GS,MP,3P,3PA,eFG_pct,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,3P_pct
0,martike01,1996-97,CUSA,FR,22,3,233,0.0,0.0,0.65,...,NaN,12.7,1.7,2.4,4.1,2.7,8.8,10.6,NaN,NaN
1,martike01,1997-98,CUSA,SO,30,30,858,0.0,0.0,0.626,...,NaN,12.4,1.9,1.4,3.9,2.4,5.1,13.9,NaN,0.0
2,martike01,1998-99,CUSA,JR,33,33,900,0.0,0.0,0.573,...,NaN,10.1,2.2,1.6,3.5,3.0,4.8,14.8,NaN,0.0
3,martike01,1999-00,CUSA,SR,31,31,909,0.1,0.3,0.571,...,9.9,13.2,1.8,1.9,4.7,2.5,3.1,25.7,"AP POY-1,WA-1,AA-1,NA-1",0.286
4,swiftst01,1998-99,SEC,FR,16,7,319,0.1,1.0,0.414,...,NaN,8.7,0.6,1.9,4.4,5.8,5.4,15.2,NaN,0.125


In [24]:
# 1. Preparar o DataFrame de Carreira (Soma dos Totais)
# Agrupamos por ID e somamos apenas as colunas de "Volume"
df_career_totals = df_college_per_min_init.groupby('Player_ID').agg({
    'G': 'sum',
    'GS': 'sum',
    'MP': 'sum',
    'Awards': 'count' 
}).reset_index()

df_career_totals.rename(columns={
    'G': 'Career_Games',
    'GS': 'Career_Starts',
    'MP': 'Career_Minutes',
    'Awards': 'Career_Awards_Count'
}, inplace=True)

In [25]:
# 2. Preparar o DataFrame da Última Temporada (Habilidade Atual)
# Ordena por jogador e ano, e pega a última linha
df_last_season = df_college_per_min_init.sort_values(['Player_ID', 'Season']).drop_duplicates('Player_ID', keep='last')

cols_performance = [
    'Player_ID', 'Season', 'Conf', 'Class', # Identificadores
    '3P', '3PA', 'eFG_pct', 'FTA', 'FT_pct',             # Arremesso
    'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', '3P_pct' # Stats Puras
]
df_last_season = df_last_season[cols_performance]

In [26]:
# 3. Juntar tudo no Dataset Final do Modelo
df_college_per_min = pd.merge(df_last_season, df_career_totals, on='Player_ID', how='left')

# 4. Converter 'Class' para 'Years_College'
df_college_per_min['Years_College'] = df_college_per_min['Class'].apply(convert.class_to_int)

# 5. Drop Column: Remove coluna 'Season' que não é mais necessária
df_college_per_min.drop(['Season', 'Class'], axis=1, inplace=True)

In [27]:
df_college_per_min.rename(columns={
    'Conf': 'Conference',
    '3P': 'Three_Pointers_Made',
    '3PA': 'Three_Pointers_Attempted',
    'eFG_pct': 'Effective_Field_Goal_Pct',
    'FTA': 'Free_Throw_Attempted',
    'FT_pct': 'Free_Throw_Pct',
    'ORB': 'Offensive_Rebounds',
    'DRB': 'Defensive_Rebounds',
    'TRB': 'Total_Rebounds',
    'AST': 'Assists',
    'STL': 'Steals',
    'BLK': 'Blocks',
    'TOV': 'Turnovers',
    'PF': 'Personal_Fouls',
    'PTS': 'Points',
    '3P_pct': 'Three_Point_Pct'
}, inplace=True)

df_college_per_min.columns = df_college_per_min.columns.str.lower()

In [28]:
# Verificação
df_college_per_min.head()

,player_id,conference,three_pointers_made,three_pointers_attempted,effective_field_goal_pct,free_throw_attempted,free_throw_pct,offensive_rebounds,defensive_rebounds,total_rebounds,...,blocks,turnovers,personal_fouls,points,three_point_pct,career_games,career_starts,career_minutes,career_awards_count,years_college
0,achiupr01,AAC,0.6,1.7,0.511,7.9,0.599,3.9,10.2,14.2,...,2.5,3.7,3.1,20.7,0.325,31,31,943,0,1
1,ackeral01,WCC,2.0,4.8,0.522,3.7,0.858,1.5,5.5,7.0,...,0.5,2.8,2.0,17.9,0.431,282931,232631,9319401152,0,3
2,acyqu01,Big 12,0.1,0.2,0.582,6.2,0.782,3.5,6.5,10.0,...,2.5,2.5,4.1,16.1,0.6,34363138,1001838,5588409601130,0,4
3,adamsha01,Pac-10,0.7,2.5,0.495,3.7,0.611,2.6,3.4,6.0,...,0.5,2.7,2.3,21.1,0.262,32303731,5283730,57291510911031,0,4
4,adamsjo01,Pac-12,1.9,5.4,0.545,7.0,0.836,2.5,4.6,7.1,...,0.2,2.0,2.9,23.1,0.356,3336,2736,9991082,0,2


#### **Mock do Draft**

In [29]:
dfs['mock'].head()

,Year,Mock_Pick,Player,Team,Class,Pos,College,Weight,Height
0,2000,1,Kenyon Martin,New Jersey,Sr,PF,NaN,NaN,NaN
1,2000,2,Stromile Swift,Vancouver,So,PF,NaN,NaN,NaN
2,2000,3,Marcus Fizer,LA Clippers,Jr,PF,NaN,NaN,NaN
3,2000,4,Darius Miles,Chicago,Sr,SF,NaN,NaN,NaN
4,2000,5,Chris Mihm,Orlando (from GS),Jr,C,NaN,NaN,NaN


In [30]:
df_mock_draft = dfs['mock'][['Player','Year','Mock_Pick']].copy()

In [31]:
# Renomeia para não confundir depois
df_mock_draft.rename(columns={
    'Player': 'Player_Name',
    'Year': 'Draft_Year'
}, inplace=True)

df_mock_draft.columns = df_mock_draft.columns.str.lower()

In [32]:
df_mock_draft.head()

,player_name,draft_year,mock_pick
0,Kenyon Martin,2000,1
1,Stromile Swift,2000,2
2,Marcus Fizer,2000,3
3,Darius Miles,2000,4
4,Chris Mihm,2000,5


#### **Dados dos jogadores do Draft**

In [33]:
dfs['college_player_metadata'].head()

,Player_ID,Position,Height_Weight_Raw,RSCI_Rank,RSCI_Year
0,martike01,Center,"6-9, 234lb",NaN,NaN
1,swiftst01,Forward,"6-9, 225lb",5.0,1998.0
2,fizerma01,Forward,"6-9, 262lb",NaN,NaN
3,millemi01,Forward,"6-8, 218lb",13.0,1998.0
4,johnsde03,Guard,"6-9, 201lb",17.0,1999.0


In [34]:
df_college_metadata_init = dfs['college_player_metadata'][['Player_ID','Position','Height_Weight_Raw']].copy()

df_college_metadata_init.columns = df_college_metadata_init.columns.str.lower()

In [35]:
df_college_metadata_init

,player_id,position,height_weight_raw
0,martike01,Center,"6-9, 234lb"
1,swiftst01,Forward,"6-9, 225lb"
2,fizerma01,Forward,"6-9, 262lb"
3,millemi01,Forward,"6-8, 218lb"
4,johnsde03,Guard,"6-9, 201lb"
...,...,...,...
1178,tonjejo01,Guard,"6-5, 222lb"
1179,peterta01,Guard,"6-4, 185lb"
1180,richawi02,Guard,"6-5, 195lb"
1181,shulgma01,Guard,"6-4, 197lb"


In [36]:
# 1. PREPARAÇÃO: Resetar o Índice para garantir o alinhamento

tuples_series = df_college_metadata_init['height_weight_raw'].apply(
    lambda x: convert._parse_height_weight_string(x)
)

parsed_cols = pd.DataFrame(
    tuples_series.tolist(), 
    columns=['height_ft', 'height_in', 'weight_lbs']
)

In [37]:
# 2. CONCATENAÇÃO DAS BASES
df_college_metadata = pd.concat([df_college_metadata_init, parsed_cols], axis=1)

In [38]:
# 3. APLICAÇÃO DA CONVERSÃO MÉTRICA (usando df_metadata)

# A. Conversão de Altura (CM)
df_college_metadata['height_cm'] = df_college_metadata.apply(
    # Agora a lambda tem CERTEZA que as colunas existem por nome
    lambda row: convert.ft_in_to_cm(row['height_ft'], row['height_in']),
    axis=1 
)

# B. Conversão de Peso (KG)
df_college_metadata['weight_kg'] = df_college_metadata['weight_lbs'].apply(convert.lbs_to_kg)


# 4. DROP Colunas: Remoção de colunas nao utilizadas
df_college_metadata.drop(['height_weight_raw', 'height_ft','height_in','weight_lbs'], axis=1, inplace=True)

In [39]:
df_college_metadata.head()

,player_id,position,height_cm,weight_kg
0,martike01,Center,205.7,106.1
1,swiftst01,Forward,205.7,102.1
2,fizerma01,Forward,205.7,118.8
3,millemi01,Forward,203.2,98.9
4,johnsde03,Guard,205.7,91.2


#### **Dados Antropométricos do jogadores do Draft**

In [40]:
dfs['combine_anthro'].head()

,TEMP_PLAYER_ID,PLAYER_ID,FIRST_NAME,LAST_NAME,PLAYER_NAME,POSITION,HEIGHT_WO_SHOES,HEIGHT_WO_SHOES_FT_IN,HEIGHT_W_SHOES,HEIGHT_W_SHOES_FT_IN,WEIGHT,WINGSPAN,WINGSPAN_FT_IN,STANDING_REACH,STANDING_REACH_FT_IN,BODY_FAT_PCT,HAND_LENGTH,HAND_WIDTH,Season_Year
0,NaN,2124,Malik,Allen,Malik Allen,PF-C,80.25,6' 8.25'',NaN,NaN,271.0,86.5,7' 2.5'',109.0,9' 1'',NaN,NaN,NaN,2000
1,NaN,12019,Harold,Arceneaux,Harold Arceneaux,SG-SF,76.50,6' 4.5'',NaN,NaN,219.0,80.5,6' 8.5'',103.0,8' 7'',NaN,NaN,NaN,2000
2,NaN,12020,Lamont,Barnes,Lamont Barnes,PF-C,80.50,6' 8.5'',NaN,NaN,235.5,87.5,7' 3.5'',108.0,9' 0'',NaN,NaN,NaN,2000
3,NaN,12131,Mario,Bland,Mario Bland,PF,77.50,6' 5.5'',NaN,NaN,287.0,84.0,7' 0'',103.0,8' 7'',NaN,NaN,NaN,2000
4,NaN,2056,Primoz,Brezec,Primoz Brezec,C,84.75,7' 0.75'',NaN,NaN,243.0,86.0,7' 2'',110.0,9' 2'',NaN,NaN,NaN,2000


In [41]:
df_combine_anthro_init = dfs['combine_anthro'][['PLAYER_NAME','Season_Year','HEIGHT_WO_SHOES','WEIGHT','WINGSPAN', 'STANDING_REACH', 'BODY_FAT_PCT']].copy()

df_combine_anthro_init.columns = df_combine_anthro_init.columns.str.lower()

In [42]:
df_combine_anthro_init.loc[:, 'has_combine_anthro'] = 1
df_combine_anthro_init['height_cm'] = df_combine_anthro_init['height_wo_shoes'].apply(convert.inches_to_cm)
df_combine_anthro_init['wingspan_cm'] = df_combine_anthro_init['wingspan'].apply(convert.inches_to_cm)
df_combine_anthro_init['standing_reach_cm'] = df_combine_anthro_init['standing_reach'].apply(convert.inches_to_cm)
df_combine_anthro_init['weight_kg'] = df_combine_anthro_init['weight'].apply(convert.lbs_to_kg)

In [43]:
df_combine_anthro = df_combine_anthro_init.copy()

df_combine_anthro.drop(['height_wo_shoes', 'wingspan','standing_reach','weight'], axis=1, inplace=True)

df_combine_anthro.rename(columns={
    'player_name': 'player_name',
    'body_fat_pct': 'body_fat_pct'
    }, inplace=True)

In [44]:
df_combine_anthro.head()

,player_name,season_year,body_fat_pct,has_combine_anthro,height_cm,wingspan_cm,standing_reach_cm,weight_kg
0,Malik Allen,2000,NaN,1,203.8,219.7,276.9,122.9
1,Harold Arceneaux,2000,NaN,1,194.3,204.5,261.6,99.3
2,Lamont Barnes,2000,NaN,1,204.5,222.2,274.3,106.8
3,Mario Bland,2000,NaN,1,196.8,213.4,261.6,130.2
4,Primoz Brezec,2000,NaN,1,215.3,218.4,279.4,110.2


#### **Dados Agilidade e Força do jogadores do Draft**

In [45]:
dfs['combine_strength_agility'].head()

KeyError: 'combine_strength_agility'

In [ ]:
df_combine_strength_agility_init = dfs['combine_strength_agility'][['PLAYER_NAME','Season_Year','MAX_VERTICAL_LEAP','LANE_AGILITY_TIME','THREE_QUARTER_SPRINT', 'BENCH_PRESS']].copy()

df_combine_strength_agility_init.columns = df_combine_strength_agility_init.columns.str.lower()

In [ ]:
df_combine_strength_agility_init.head()

,player_name,season_year,max_vertical_leap,lane_agility_time,three_quarter_sprint,bench_press
0,Rawle Alkins,2018,40.5,11.50,3.15,10.0
1,Grayson Allen,2018,40.5,10.31,3.15,NaN
2,Kostas Antetokounmpo,2018,35.0,12.48,3.21,0.0
3,Udoka Azubuike,2018,31.5,12.97,3.12,11.0
4,Mohamed Bamba,2018,NaN,NaN,NaN,NaN


In [ ]:
df_combine_strength_agility_init['has_combine_strength_agility'] = df_combine_strength_agility_init['max_vertical_leap'].notna().astype(int)

df_combine_strength_agility_init['max_vertical_leap_cm'] = df_combine_strength_agility_init['max_vertical_leap'].apply(convert.inches_to_cm)

In [ ]:
df_combine_strength_agility = df_combine_strength_agility_init.copy()

df_combine_strength_agility.drop(['max_vertical_leap'], axis=1, inplace=True)

df_combine_strength_agility.rename(columns={
    'lane_agility_time': 'lane_agility_time_sec',
    'three_quarter_sprint': 'three_quarter_sprint_sec',
    'bench_press': 'bench_press_reps'
    }, inplace=True)

In [ ]:
df_combine_strength_agility.head()

,player_name,season_year,lane_agility_time_sec,three_quarter_sprint_sec,bench_press_reps,has_combine_strength_agility,max_vertical_leap_cm
0,Rawle Alkins,2018,11.50,3.15,10.0,1,102.9
1,Grayson Allen,2018,10.31,3.15,NaN,1,102.9
2,Kostas Antetokounmpo,2018,12.48,3.21,0.0,1,88.9
3,Udoka Azubuike,2018,12.97,3.12,11.0,1,80.0
4,Mohamed Bamba,2018,NaN,NaN,NaN,0,NaN


### **Merge das tabelas**

#### **Joins com player_id**

In [46]:
df_final_dataset = (
    df_draft
    .merge(df_nba_stats, on=['player_id'], how='left')
    .merge(df_college_adv, on='player_id', how='left')
    .merge(df_college_per_min, on='player_id', how='left')
    .merge(df_college_metadata, on='player_id', how='left')
)

#### **Joins com player_name**

In [47]:
def apply_fuzzy_match_by_year(df_target, df_candidate, 
                              target_year_col, candidate_year_col, 
                              target_name_col, candidate_name_col, 
                              new_join_col_name, cutoff=85):
    """
    Realiza fuzzy matching ano a ano para evitar falsos positivos entre jogadores de épocas diferentes.
    Atualiza o DataFrame df_target criando uma coluna de chave de junção.
    """
    # Inicializa a coluna de join com o próprio nome (caso não ache match melhor, usa o original)
    df_target[new_join_col_name] = df_target[target_name_col]
    
    # Pega os anos comuns entre as duas bases
    years_target = set(df_target[target_year_col].unique())
    years_candidate = set(df_candidate[candidate_year_col].unique())
    common_years = years_target.intersection(years_candidate)
    
    print(f"Processando fuzzy match para {len(common_years)} anos em comum...")

    for year in sorted(common_years):
        # 1. Filtra os nomes APENAS daquele ano
        target_names_year = df_target[df_target[target_year_col] == year][target_name_col].unique()
        candidate_names_year = df_candidate[df_candidate[candidate_year_col] == year][candidate_name_col].unique()
        
        # 2. Cria o mapa de nomes para este ano específico
        year_mapping = {}
        for name in target_names_year:
            match = features.find_best_match(name, candidate_names_year, cutoff=cutoff)
            if match:
                year_mapping[name] = match
        
        # 3. Atualiza o DataFrame APENAS nas linhas deste ano
        # Se encontrou match no dicionário, usa ele. Se não, mantém o valor que já estava.
        if year_mapping:
            mask = df_target[target_year_col] == year
            df_target.loc[mask, new_join_col_name] = df_target.loc[mask, target_name_col].map(year_mapping).fillna(df_target.loc[mask, target_name_col])

    return df_target

In [48]:
df_draft_merged_pn = df_draft.copy()
df_mock_draft_merged_pn = df_mock_draft.copy()
df_combine_anthro_merged_pn = df_combine_anthro.copy()

##### **Draft x Mock**

In [49]:
# Limpeza inicial dos nomes
df_draft_merged_pn['name_clean'] = df_draft_merged_pn['player_name'].apply(convert.clean_name)
df_mock_draft_merged_pn['name_clean'] = df_mock_draft_merged_pn['player_name'].apply(convert.clean_name)

# Aplica a lógica ano a ano
df_draft_merged_pn = apply_fuzzy_match_by_year(
    df_target=df_draft_merged_pn,
    df_candidate=df_mock_draft_merged_pn,
    target_year_col='draft_year',
    candidate_year_col='draft_year', # Confirme se o nome da coluna é esse no mock
    target_name_col='name_clean',
    candidate_name_col='name_clean',
    new_join_col_name='join_key_mock',
    cutoff=80
)

# Merge usando a chave gerada e o ANO (segurança dupla)
df_mock_draft_merged_pn = pd.merge(
    df_draft_merged_pn,
    df_mock_draft_merged_pn[['name_clean', 'draft_year', 'mock_pick']], 
    left_on=['join_key_mock', 'draft_year'],
    right_on=['name_clean', 'draft_year'],
    how='left',
    suffixes=('', '_mock')
)

# Adiciona coluna ao dataset final
cols_to_add_mock = df_mock_draft_merged_pn[['player_id', 'mock_pick']].copy()
df_final_dataset.drop(columns=['mock_pick'], inplace=True, errors='ignore')
df_final_dataset = pd.merge(df_final_dataset, cols_to_add_mock, on='player_id', how='left')

Processando fuzzy match para 26 anos em comum...


##### **Draft x Nba Draft Compose**

In [50]:
# 1. Preparação e Fuzzy Match (Mantido igual)
df_combine_anthro_merged_pn['name_clean'] = df_combine_anthro_merged_pn['player_name'].apply(convert.clean_name)

df_draft_merged_pn = apply_fuzzy_match_by_year(
    df_target=df_draft_merged_pn,
    df_candidate=df_combine_anthro_merged_pn,
    target_year_col='draft_year',
    candidate_year_col='season_year',
    target_name_col='name_clean',
    candidate_name_col='name_clean',
    new_join_col_name='join_key_combine',
    cutoff=80
)

# 2. Merge Intermediário para pegar os dados do Combine
# Note o uso de suffixes para identificar o que veio de onde se houver colisão de nomes
df_merged_combine = pd.merge(
    df_draft_merged_pn,
    df_combine_anthro_merged_pn[['name_clean', 'season_year', 'height_cm', 'weight_kg']], 
    left_on=['join_key_combine', 'draft_year'],
    right_on=['name_clean', 'season_year'],
    how='left',
    suffixes=('', '_combine_raw') # adiciona sufixo se a coluna já existir no df_draft
)

# 3. Seleciona as colunas para levar ao Dataset Final
# Renomeamos Height e Weight explicitamente para não colidir com o que já existe no df_final_dataset
cols_to_add = df_merged_combine[[
    'player_id', 
    'height_cm',      # altura do combine (pode virar height_cm_combine no merge abaixo)
    'weight_kg'     # peso do combine
]].copy()

cols_to_add.rename(columns={
    'height_cm': 'height_cm_combine',
    'weight_kg': 'weight_kg_combine'
}, inplace=True)

# 4. Merge com o Dataset Final
df_final_dataset = pd.merge(df_final_dataset, cols_to_add, on='player_id', how='left')

# --- 5. APLICAÇÃO DA REGRA DE NEGÓCIO (Combine > College) ---

print("Aplicando regra de prioridade: NBA Combine > College Stats...")

# Altura: Preenche a coluna Height_cm_combine com o valor do College onde estiver vazio. 
# Se tiver valor no Combine, ele prevalece.
# Depois, sobrescrevemos a coluna oficial 'Height_cm'.
df_final_dataset['height_cm'] = df_final_dataset['height_cm_combine'].fillna(df_final_dataset['height_cm'])

# Peso: Mesma lógica
df_final_dataset['weight_kg'] = df_final_dataset['weight_kg_combine'].fillna(df_final_dataset['weight_kg'])

# 6. Limpeza: Remove as colunas temporárias de suporte
df_final_dataset.drop(columns=['height_cm_combine', 'weight_kg_combine'], inplace=True)

Processando fuzzy match para 26 anos em comum...
Aplicando regra de prioridade: NBA Combine > College Stats...


##### **Completar valores Nulos ou Vazios**

In [51]:
print(len(df_final_dataset[df_final_dataset['mock_pick'].isna()]))
print(df_final_dataset.duplicated(subset=['player_id']).sum())
print(len(df_final_dataset[df_final_dataset['college_name'].isna()]))

306
0
348


In [ ]:
df_final_dataset['vorp_5y_sum'] = df_final_dataset['vorp_5y_sum'].fillna(0)
df_final_dataset['mock_pick'] = df_final_dataset['mock_pick'].fillna(61) ##Jogadores sem dados no Mock recebem pick 61
df_final_dataset['seasons_played'] = df_final_dataset['seasons_played'].fillna(0)


In [64]:
df_final_dataset.to_csv(PROJECT_ROOT + "/data/final_dataset.csv", index=False, encoding="utf-8-sig")

In [53]:

# 3. Criar a Variável Binária (o seu 'y')
# Por exemplo, sucesso é ter um VORP Acumulado > 1.5 nos 5 primeiros anos.
# VORP_THRESHOLD = 1.5
# vorp_target['Success'] = (vorp_target['VORP_5Y_SUM'] > VORP_THRESHOLD).astype(int)

# 4. Merge Final: Juntar a métrica de sucesso de volta à sua base mestra
# df_base_mestra = pd.merge(df_base_mestra, vorp_target[['Player_ID', 'Success']], on='Player_ID', how='left')


In [54]:
# # Define o ano limite
# LAST_VALID_DRAFT = 2020

# # Cria o DataFrame de Treino (apenas jogadores que já completaram 5 anos)
# df_draft = df_draft[df_draft['Year'] <= LAST_VALID_DRAFT].copy()

##### **Outros**

In [55]:
# df_draft['name_clean'] = df_draft['player_name'].apply(convert.clean_name)
# df_mock_draft['name_clean'] = df_mock_draft['player_name'].apply(convert.clean_name)
# draft_names = df_draft['name_clean'].unique()
# choices = df_mock_draft['name_clean'].unique()

In [56]:
# name_mapping = {}

# for name in draft_names:
#     # Tenta achar na lista do draft
#     match = features.find_best_match(name, choices, cutoff=80) # 80% de similaridade
#     if match:
#         name_mapping[name] = match

# df_draft['join_key'] = df_draft['name_clean'].map(name_mapping).fillna(df_draft['name_clean'])

# df_join_key_mock = pd.merge(
#         df_draft, 
#         df_mock_draft, 
#         left_on='join_key', 
#         right_on='name_clean', 
#         how='left'
#     )

# df_join_key_mock = df_join_key_mock[['player_id','mock_pick']].copy()

In [57]:
# df_final_dataset = pd.merge(
#     df_final_dataset, 
#     df_join_key_mock, 
#     on='player_id',
#     how='left'
# )

In [58]:
# df_draft['name_clean'] = df_draft['player_name'].apply(convert.clean_name)
# df_mock_draft['name_clean'] = df_mock_draft['player_name'].apply(convert.clean_name)
# df_combine_anthro['name_clean'] = df_combine_anthro['player_name'].apply(convert.clean_name)

In [59]:
# draft_to_mock_map = features.get_fuzzy_mapping(
#     df_draft['name_clean'].unique(), 
#     df_mock_draft['name_clean'].unique()
# )

In [60]:
# # Cria a chave de junção
# df_draft['join_key_mock'] = df_draft['name_clean'].map(draft_to_mock_map)

# # Faz o Merge
# df_merged_mock = pd.merge(
#     df_draft,
#     df_mock_draft[['name_clean', 'mock_pick']], # Traz apenas o que importa
#     left_on='join_key_mock',
#     right_on='name_clean',
#     how='left',
#     suffixes=('', '_mock')
# )

# # Seleciona colunas finais do Mock para o dataset principal
# cols_to_add_mock = df_merged_mock[['player_id', 'mock_pick']].copy()

# # Junta ao Dataset Final
# df_final_dataset = pd.merge(df_final_dataset, cols_to_add_mock, on='player_id', how='left')


In [61]:
# --- 3. Merge: Draft x NBA Combine ---
print("Realizando Fuzzy Match: Draft -> Combine...")
# Mapeia nomes do Draft -> Nomes do Combine
draft_to_combine_map = features.get_fuzzy_mapping(
    df_draft['name_clean'].unique(), 
    df_combine_anthro['name_clean'].unique()
)

df_draft['join_key_combine'] = df_draft['name_clean'].map(draft_to_combine_map)

# Faz o Merge
df_merged_combine = pd.merge(
    df_draft,
    df_combine_anthro[['name_clean', 'height_cm', 'weight_kg']], 
    left_on='join_key_combine',
    right_on='name_clean',
    how='left',
    suffixes=('', '_combine')
)

# Seleciona colunas finais do Combine
cols_to_add_combine = df_merged_combine[['player_id', 'height_cm', 'weight_kg']].copy()

# Junta ao Dataset Final (CORREÇÃO CRÍTICA AQUI)
df_final_dataset = pd.merge(df_final_dataset, cols_to_add_combine, on='player_id', how='left')

print("Merges concluídos.")

Realizando Fuzzy Match: Draft -> Combine...


KeyError: 'name_clean'

In [ ]:
# df_merged_mock = pd.merge(
#     df_draft,
#     df_mock_draft[['name_clean', 'mock_pick']], # Traz apenas o que importa
#     left_on='join_key_mock',
#     right_on='name_clean',
#     how='left'
# )

In [ ]:
# df_draft['name_clean'] = df_draft['player_name'].apply(convert.clean_name)
# df_combine_anthro['name_clean'] = df_combine_anthro['player_name'].apply(convert.clean_name)
# draft_names = df_draft['name_clean'].unique()
# choices = df_combine_anthro['name_clean'].unique()

In [ ]:
# name_mapping = {}

# for name in draft_names:
#     # Tenta achar na lista do draft
#     match = features.find_best_match(name, choices, cutoff=80) # 80% de similaridade
#     if match:
#         name_mapping[name] = match

# df_draft['join_key'] = df_draft['name_clean'].map(name_mapping).fillna(df_draft['name_clean'])

# df_join_key_nba_combine = pd.merge(
#         df_draft, 
#         df_combine_anthro, 
#         left_on='join_key', 
#         right_on='name_clean', 
#         how='left'
#     )

# df_join_key_nba_combine = df_join_key_nba_combine[['player_id','height_cm','weight_kg']].copy()

In [ ]:
# df_final_dataset = pd.merge(
#     df_final_dataset, 
#     df_join_key_mock, 
#     on='player_id',
#     how='left'
# )

#### **Outros**

teste

In [ ]:

# 3. Criar a Variável Binária (o seu 'y')
# Por exemplo, sucesso é ter um VORP Acumulado > 1.5 nos 5 primeiros anos.
# VORP_THRESHOLD = 1.5
# vorp_target['Success'] = (vorp_target['VORP_5Y_SUM'] > VORP_THRESHOLD).astype(int)

# 4. Merge Final: Juntar a métrica de sucesso de volta à sua base mestra
# df_base_mestra = pd.merge(df_base_mestra, vorp_target[['Player_ID', 'Success']], on='Player_ID', how='left')